<a href="https://colab.research.google.com/github/aaohl-lanlan/ia-ciberseguranca/blob/main/ia_ciberseguranca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA para Cibersegurança — Setup do Projeto (Aula 1)


**Dupla:** `Allan Accioly `  ·  `Lucas Chaves`

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path

# Raiz do projeto no Drive (ajuste o nome se quiser)
PROJETO = Path('/content/drive/MyDrive/ia-ciberseguranca')
SUBDIRS = ['data/raw', 'data/processed', 'notebooks', 'src', 'models', 'reports']

for d in SUBDIRS:
    (PROJETO / d).mkdir(parents=True, exist_ok=True)

print('Projeto em:', PROJETO)
!ls -la "{PROJETO}"

Projeto em: /content/drive/MyDrive/ia-ciberseguranca
total 20
drwx------ 4 root root 4096 Aug 13 13:23 data
drwx------ 2 root root 4096 Aug 13 13:23 models
drwx------ 2 root root 4096 Aug 13 13:23 notebooks
drwx------ 2 root root 4096 Aug 13 13:23 reports
drwx------ 2 root root 4096 Aug 13 13:23 src


In [3]:
import sys, platform
from importlib.metadata import version, PackageNotFoundError

print('Python :', sys.version.split()[0], '|', platform.platform())
print('-' * 40)

PACOTES = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib',
           'seaborn', 'imbalanced-learn', 'xgboost']
for p in PACOTES:
    try:
        print(f'{p:18s} {version(p)}')
    except PackageNotFoundError:
        print(f'{p:18s} (não instalado)')

Python : 3.12.13 | Linux-6.6.122+-x86_64-with-glibc2.35
----------------------------------------
numpy              2.0.2
pandas             2.2.2
scikit-learn       1.6.1
scipy              1.16.3
matplotlib         3.10.0
seaborn            0.13.2
imbalanced-learn   0.14.2
xgboost            3.3.0


In [4]:
%pip install -q imbalanced-learn xgboost

In [5]:
from importlib.metadata import version

PKGS = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib',
        'seaborn', 'imbalanced-learn', 'xgboost']

req = PROJETO / 'requirements.txt'
req.write_text('\n'.join(f'{p}=={version(p)}' for p in PKGS) + '\n')

print(req, 'gerado:\n')
print(req.read_text())

/content/drive/MyDrive/ia-ciberseguranca/requirements.txt gerado:

numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
scipy==1.16.3
matplotlib==3.10.0
seaborn==0.13.2
imbalanced-learn==0.14.2
xgboost==3.3.0



In [6]:
import os, random
import numpy as np

def set_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ModuleNotFoundError:
        pass

set_seed(42)
print('Seeds fixadas em 42')

Seeds fixadas em 42


In [7]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

X, y = make_classification(n_samples=10_000, n_features=20, n_informative=6,
                           weights=[0.98, 0.02], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f'Positivos no teste : {y_te.mean():.2%}')
print(f'ROC-AUC            : {roc_auc_score(y_te, proba):.3f}')
print(f'PR-AUC             : {average_precision_score(y_te, proba):.3f}   <- a que importa aqui')

Positivos no teste : 2.53%
ROC-AUC            : 0.791
PR-AUC             : 0.547   <- a que importa aqui
